# 🎙️ Dhvani Kannada F5-TTS Training (Google Colab)

### ⚡ Prerequisites:
1. In the top menu, select: **Runtime → Change runtime type → T4 GPU**.
2. Run the cells step-by-step to fine-tune your custom voice.

### Step 1: Check GPU & Clone Repository

In [ ]:
!nvidia-smi
!git clone --depth 1 https://github.com/karthik7026/dhvani-kannada-tts.git /content/dhvani
%cd /content/dhvani
!pip -q install -r requirements.txt
!git clone --depth 1 https://github.com/SWivid/F5-TTS.git /content/F5-TTS
!pip -q install -e /content/F5-TTS
!pip -q install soundfile torchaudio librosa

### Step 2: Load Training Dataset
You can either **upload `dhvani-training-data.zip` directly** or mount Google Drive.

In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path

# If dhvani-training-data.zip is in the repo, extract it directly
repo_zip = Path('/content/dhvani/training/dhvani-training-data.zip')
data_dir = Path('/content/dhvani/training/data')

if repo_zip.exists() and not (data_dir / 'metadata_f5.csv').exists():
    with zipfile.ZipFile(repo_zip, 'r') as zip_ref:
        zip_ref.extractall('/content/dhvani/training')
    print('Extracted dataset from repository zip.')

# Generate F5 Manifest
subprocess.run(['python3', 'training/export_f5_manifest.py', str(data_dir), '/content/metadata_f5.csv'], check=True)
print('✓ Training manifest validated and prepared at /content/metadata_f5.csv')

### Step 3: Prepare Vocab & Start F5-TTS Fine-Tuning

In [ ]:
%cd /content/F5-TTS
!python src/f5_tts/train/datasets/prepare_csv_wavs.py /content/metadata_f5.csv data/dhvani_kn_custom --pretrain

# Start Fine-Tuning on T4 GPU
!f5-tts_finetune-cli \
    --exp_name F5TTS_v1_Base \
    --dataset_name dhvani_kn_custom \
    --tokenizer custom \
    --tokenizer_path data/dhvani_kn_custom/vocab.txt \
    --finetune \
    --epochs 15 \
    --batch_size_per_gpu 350 \
    --batch_size_type frame \
    --max_samples 16 \
    --grad_accumulation_steps 8 \
    --save_per_updates 200 \
    --keep_last_n_checkpoints 3

### Step 4: Test Synthesis with Trained Checkpoint

In [ ]:
# Run inference test with the latest checkpoint
!f5-tts_infer-cli \
    --model F5TTS_v1_Base \
    --ckpt_file /content/F5-TTS/ckpts/F5TTS_v1_Base/model_last.pt \
    --gen_text "ನಮಸ್ಕಾರ! ಧ್ವನಿ ಕನ್ನಡ ಆಡಿಯೊ ಸ್ಟುಡಿಯೋಗೆ ಸುಸ್ವಾಗತ." \
    --output_dir /content/output

from IPython.display import Audio
import glob
wavs = glob.glob('/content/output/*.wav')
if wavs:
    display(Audio(wavs[-1]))